In [1]:
# ============================================================
# BLOCK 1: Import required libraries
# ============================================================
# ✅ (No change — just ensure dependencies are installed)

%pip install requests pandas
import numpy as np
import requests
import pandas as pd
import time
import os
from datetime import datetime
from requests.exceptions import RequestException, JSONDecodeError

# ============================================================
# BLOCK 2: Define storage file for transactions
# ============================================================
# ✅ (No change)
DATA_FILE = "mempool_log.csv"


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# BLOCK 3: Fetch ancestor transactions (Improved with Error Handling)
# ============================================================
def fetch_ancestors(txid):
    """
    Fetches details of a transaction from mempool.space API.
    Returns a list of parent txids (if available).
    """
    url = f"https://mempool.space/api/tx/{txid}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        try:
            data = response.json()
        except JSONDecodeError:
            print(f"⚠️ JSON error while decoding ancestor data for {txid}")
            return []

        parents = [vin.get("txid") for vin in data.get("vin", []) if "txid" in vin]
        return parents if parents else []
    except RequestException as e:
        print(f"⚠️ Network/API error for {txid}: {e}")
        return []
    except Exception as e:
        print(f"⚠️ Unexpected error fetching ancestor for {txid}: {e}")
        return []


In [3]:
# ============================================================
# BLOCK 4: Fetch mempool transactions (Improved error handling)
# ============================================================
def fetch_mempool_data():
    url = "https://mempool.space/api/mempool/recent"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        try:
            return response.json()
        except JSONDecodeError:
            print("⚠️ JSON decoding error while reading mempool data.")
            return []
    except RequestException as e:
        print(f"⚠️ Network error while fetching mempool: {e}")
        return []
    except Exception as e:
        print(f"⚠️ Unexpected error: {e}")
        return []


In [4]:
# ============================================================
# BLOCK 5: Log mempool data to CSV (with safety checks)
# ============================================================
def log_mempool_data():
    """
    Fetches live mempool transactions, adds timestamps + ancestor info,
    and logs into CSV safely.
    """
    txs = fetch_mempool_data()
    if not txs:
        print("⚠️ No transactions fetched this cycle.")
        return

    now = datetime.utcnow()
    rows = []

    for tx in txs:
        try:
            txid = tx.get("txid", "unknown")
            fee = tx.get("fee", 0) or 0
            vsize = tx.get("vsize", 0) or 1  # avoid division by zero
            ancestors = fetch_ancestors(txid)

            rows.append({
                "txid": txid,
                "fee": fee,
                "vsize": vsize,
                "time": now,
                "ancestors": ";".join(ancestors) if ancestors else "None"
            })
        except Exception as e:
            print(f"⚠️ Error logging transaction {tx.get('txid', 'unknown')}: {e}")

    df = pd.DataFrame(rows)
    try:
        if not os.path.exists(DATA_FILE):
            df.to_csv(DATA_FILE, index=False)
        else:
            df.to_csv(DATA_FILE, mode="a", header=False, index=False)
        print(f"✅ Logged {len(rows)} transactions at {now}")
    except Exception as e:
        print(f"⚠️ CSV write error: {e}")


In [ ]:
# ============================================================
# BLOCK 6: Run logger loop with network and manual interruption safety
# ============================================================
for i in range(60):  # run 60 cycles (~1 hour if 60 sec delay)
    try:
        print(f"\n🔁 Cycle {i+1}/60")
        log_mempool_data()
    except KeyboardInterrupt:
        print("🛑 Logging manually stopped by user.")
        break
    except Exception as e:
        print(f"⚠️ Error in logging cycle {i+1}: {e}")
    time.sleep(60)  # delay between each fetch



🔁 Cycle 1/60


C:\Users\PRAVESH\AppData\Local\Temp\ipykernel_24300\1030492874.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


✅ Logged 10 transactions at 2025-10-13 13:56:36.530609

🔁 Cycle 2/60
✅ Logged 10 transactions at 2025-10-13 13:57:42.309223

🔁 Cycle 3/60
✅ Logged 10 transactions at 2025-10-13 13:58:47.702075

🔁 Cycle 4/60
✅ Logged 10 transactions at 2025-10-13 13:59:53.457262

🔁 Cycle 5/60
✅ Logged 10 transactions at 2025-10-13 14:00:59.337553

🔁 Cycle 6/60
✅ Logged 10 transactions at 2025-10-13 14:02:04.727557

🔁 Cycle 7/60
✅ Logged 10 transactions at 2025-10-13 14:03:11.002667

🔁 Cycle 8/60
✅ Logged 10 transactions at 2025-10-13 14:04:17.428700

🔁 Cycle 9/60
✅ Logged 10 transactions at 2025-10-13 14:05:23.142250

🔁 Cycle 10/60
✅ Logged 10 transactions at 2025-10-13 14:06:28.682155

🔁 Cycle 11/60
✅ Logged 10 transactions at 2025-10-13 14:07:34.408825

🔁 Cycle 12/60
✅ Logged 10 transactions at 2025-10-13 14:08:39.778233

🔁 Cycle 13/60
✅ Logged 10 transactions at 2025-10-13 14:09:45.112121

🔁 Cycle 14/60
✅ Logged 10 transactions at 2025-10-13 14:10:50.602486

🔁 Cycle 15/60
✅ Logged 10 transactions at 

In [ ]:
# ============================================================
# BLOCK 7: Load and verify CSV
# ============================================================
try:
    df = pd.read_csv(DATA_FILE, on_bad_lines='skip', encoding='utf-8')
    if df.empty:
        print("⚠️ CSV exists but no data yet.")
    else:
        print(f"✅ CSV loaded successfully with {len(df)} rows.")
except Exception as e:
    print("❌ Error reading CSV:", e)


In [ ]:
# ---------------------------------------------
# 8️⃣ Apply improved fee scoring formula
# ---------------------------------------------

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Fill missing values with safe defaults
df["fee"] = df["fee"].fillna(0)
df["vsize"] = df["vsize"].replace(0, 1)

# Simulate ancestor fee/vsize (API doesn’t provide them directly)
df["ancestor_fee"] = np.random.choice([0, 1000, 2000, 3000], len(df))
df["ancestor_vsize"] = np.random.choice([0, 200, 400, 600], len(df))
df["ancestor_vsize"] = df["ancestor_vsize"].replace(0, np.nan)
df["ancestor_ratio"] = (df["ancestor_fee"] / df["ancestor_vsize"]).fillna(0)

# Compute main features
df["fee_per_byte"] = df["fee"] / df["vsize"]

# Normalize for stability
df["fee_norm"] = (df["fee_per_byte"] - df["fee_per_byte"].min()) / (df["fee_per_byte"].max() - df["fee_per_byte"].min() + 1e-9)
df["ancestor_norm"] = (df["ancestor_ratio"] - df["ancestor_ratio"].min()) / (df["ancestor_ratio"].max() - df["ancestor_ratio"].min() + 1e-9)

# Weights
α, β, γ = 0.6, 0.25, 0.15

# Final formula
df["score_raw"] = (α * df["fee_norm"]) + (β * np.log(df["age_min"] + 1)) + (γ * df["ancestor_norm"])
df["score"] = sigmoid(df["score_raw"])

# Sort by highest score
df = df.sort_values("score", ascending=False).reset_index(drop=True)

print("✅ Transaction scores calculated successfully!")
df.head()


In [ ]:
# ---------------------------------------------
# 9️⃣ Save evaluated dataset for analysis
# ---------------------------------------------
OUTPUT_FILE = "evaluated_mempool.csv"

try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ Saved evaluated data to {OUTPUT_FILE}")
except Exception as e:
    print("❌ Error saving evaluated dataset:", e)
